<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Solution: Time and ranking

**[Time and ranking](https://learning.nextia-ai.com/courses/sql/m03/time-and-ranking/)** · SQL and Data Preparation for AI · Module 3, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** filter tickets by date without losing the last day, and use window functions (`ROW_NUMBER`, `RANK`, `LAG`) to find each ticket's latest outcome event and to put each customer's tickets in order, with ties handled.

| | |
|---|---|
| **Time** | About 55 minutes |
| **Needs** | An internet connection for the first cell. No account and no install: only Python's standard library, and pandas to show tables if it is there (it is in Colab and Kaggle). Window functions need SQLite 3.25 or later: every current Python has it. |
| **You should know** | CTE steps and step counts, from [Readable queries](https://learning.nextia-ai.com/courses/sql/m03/readable-queries/). The keys of `tickets` and `outcomes`, and the repeated rows, from [Join tables](https://learning.nextia-ai.com/courses/sql/m02/join-tables/). |
| **You do not need** | The local project from Module 1. The setup below builds the same database here. |
| **Tested** | 2026-10-07, Python 3.14.6 (SQLite 3.50.4), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m03/time-and-ranking/) has the full explanations, the partition table and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your query between the triple quotes `"""`, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M03-L02-time-and-ranking/time-and-ranking-solution.ipynb).

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Dates are text in SQLite

SQLite has no date type. Larkfield's export stores every time as text in one format, `YYYY-MM-DD HH:MM:SS`. The parts go from the largest unit to the smallest, so the alphabetical order is also the time order. `<`, `>` and `MAX` work on it.

Four functions read this text. Run the cell: it applies them to ticket T-10001.

In [ ]:
sql("""
SELECT
    created_at,
    date(created_at) AS day,
    strftime('%Y-%m', created_at) AS month,
    datetime(created_at, '+72 hours') AS plus_72h,
    ROUND(julianday(created_at), 2) AS day_number
FROM tickets
WHERE ticket_id = 'T-10001';
""")

> **Check.** You should see `2026-01-05 07:30:24`, then `2026-01-05`, `2026-01`, `2026-01-08 07:30:24` and the day number `2461045.81` (rounded here to two decimals). The difference of two day numbers is a number of days.

The model in this course predicts an escalation **within 72 hours** of a ticket's creation. `datetime` gives that deadline. Ticket T-10008 has two outcome events. SQLite shows true as 1 and false as 0.

> **Predict.** Is each event within 72 hours? Then run the cell.

In [ ]:
sql("""
SELECT
    t.ticket_id,
    t.created_at,
    datetime(t.created_at, '+72 hours') AS deadline,
    o.recorded_at,
    o.status,
    o.recorded_at <= datetime(t.created_at, '+72 hours') AS within_72h
FROM tickets AS t
JOIN outcomes AS o
    ON o.ticket_id = t.ticket_id
WHERE t.ticket_id = 'T-10008';
""")

> **Check.** You should see two rows. The deadline is `2026-01-09 08:14:45`. The escalation came about 4 hours after creation, so `within_72h` is 1.

> **Your turn: change one thing.** In the cell above, change `'+72 hours'` to `'+2 hours'` (two places), and run it again. Now both events are after the deadline: `within_72h` is 0.

### First, check the range

Before you filter by time, look at the smallest and the largest value.

> **Predict.** The export is from 2026-07-01. What is the last `created_at`? Then run the cell.

In [ ]:
sql("""
SELECT MIN(created_at) AS first_ticket, MAX(created_at) AS last_ticket
FROM tickets;
""")

> **Check.** You should see `2026-01-05 07:30:24` and **`2036-01-24 21:27:43`**. A ticket from 2036 cannot exist. It is T-10147, a warranty ticket with a mistyped year. `MAX` does not warn you. Mei does not change the export: she selects with explicit start and end dates, and Module 4 decides what to do with the row.

### Half-open ranges

Grace, the support lead, wants the tickets of March. Tomás writes the first count with `BETWEEN`. Mei writes the second with `>=` and `<`.

> **Predict.** Are the two counts the same? Then run the cell.

In [ ]:
sql("""
SELECT
    (SELECT COUNT(*) FROM tickets
     WHERE created_at BETWEEN '2026-03-01' AND '2026-03-31') AS tomas,
    (SELECT COUNT(*) FROM tickets
     WHERE created_at >= '2026-03-01' AND created_at < '2026-04-01') AS mei;
""")

> **Check.** You should see `242` and `252`. Tomás's count misses all 10 tickets of 31 March. `BETWEEN a AND b` means `>= a AND <= b`. As text, `'2026-03-31 08:12:00'` comes after `'2026-03-31'`, because it is longer and starts with the same characters.

Mei's form is a **half-open range**: it includes the start and excludes the end. The end is the first moment that you do not want, here 1 April. It works for every time of day, months fit together with no gap and no overlap, and the same form works in PostgreSQL.

> **Your turn.** The query counts the ticket rows of May with `BETWEEN`. Run it first: 228 rows. Then change the filter to a half-open range, and run the check cell. Predict first: how many tickets were created on 31 May?

In [ ]:
may_rows = """
SELECT COUNT(*) AS may_rows
FROM tickets
WHERE created_at >= '2026-05-01'
  AND created_at <  '2026-06-01';
"""
sql(may_rows)

In [ ]:
check(may_rows, rows=1, columns=1, expected="327cddf65ac2affa")

The 4 extra rows are the tickets of 31 May. (The count is of rows: the repeated ticket rows are still in the table.)

## 2. Window functions

A `GROUP BY` turns many rows into one row per group. A **window function** keeps every row, and adds a value computed from a group of related rows, the *window*. You write it with `OVER (...)`:

```sql
ROW_NUMBER() OVER (PARTITION BY ticket_id ORDER BY recorded_at DESC, outcome_id DESC)
```

- `PARTITION BY ticket_id` splits the rows into **partitions**, one per ticket. Each partition is numbered on its own.
- `ORDER BY recorded_at DESC, outcome_id DESC` orders the rows inside each partition: the newest event first.
- `ROW_NUMBER()` gives 1, 2, 3, … Each row gets a different number, even when two rows are equal in the `ORDER BY`.

| Function | Gives | Ties (equal values in `ORDER BY`) |
|---|---|---|
| `ROW_NUMBER()` | 1, 2, 3, … | Different numbers. The database decides which tied row comes first, unless a column breaks the tie. |
| `RANK()` | 1, 2, 2, 4, … | The same number. After a tie, it skips numbers. |
| `LAG(column)` | The value in the previous row of the partition | NULL for the first row. |

The next cell numbers the events of three tickets two ways: `rn` with `ROW_NUMBER()` and `outcome_id`, and `rank_by_time` with `RANK()` by time only. The [lesson page](https://learning.nextia-ai.com/courses/sql/m03/time-and-ranking/#the-partitioned-window) shows the same rows as a table with the partitions marked.

> **Predict.** T-10136 was reopened and resolved **in the same second**. What are its `rn` and `rank_by_time` values? Then run the cell.

In [ ]:
sql("""
WITH ranked AS (
    SELECT
        ticket_id,
        outcome_id,
        recorded_at,
        status,
        ROW_NUMBER() OVER (
            PARTITION BY ticket_id
            ORDER BY recorded_at DESC, outcome_id DESC
        ) AS rn,
        RANK() OVER (
            PARTITION BY ticket_id
            ORDER BY recorded_at DESC
        ) AS rank_by_time
    FROM outcomes
)
SELECT ticket_id, outcome_id, recorded_at, status, rn, rank_by_time
FROM ranked
WHERE ticket_id IN ('T-10120', 'T-10135', 'T-10136')
ORDER BY ticket_id, rn;
""")

> **Check.** You should see 6 rows. For T-10136, `rn` is 1, 2 and 3, but `rank_by_time` is **1, 1** and 3. `recorded_at` cannot tell which event came first. `outcome_id` can: the help desk gives IDs in logging order, so O-00229 (`resolved`) was logged after O-00228 (`reopened`). The latest event is the resolution.

> **Your turn: change one thing.** Add `'T-10115'` to the list in the cell above, and run it again. Its escalation was logged twice (O-00143 and O-00144): a retry, also a tie.

## 3. Worked example: the latest outcome of each ticket

Priya, the data scientist, asks for one row per ticket with its latest status. A window function cannot go in a `WHERE` clause, because `WHERE` runs before the window is computed. So Mei uses two CTE steps: first number the rows, then keep number 1.

> **Predict.** The result must have one row per ticket. What check shows that? Then run the cell.

In [ ]:
sql("""
WITH ranked AS (
    SELECT
        ticket_id,
        outcome_id,
        recorded_at,
        status,
        ROW_NUMBER() OVER (
            PARTITION BY ticket_id
            ORDER BY recorded_at DESC, outcome_id DESC
        ) AS rn
    FROM outcomes
),
latest AS (
    SELECT ticket_id, outcome_id, recorded_at, status
    FROM ranked
    WHERE rn = 1
)
SELECT
    COUNT(*) AS latest_rows,
    COUNT(DISTINCT ticket_id) AS tickets
FROM latest;
""")

> **Check.** You should see `1399` and `1399`: one row per ticket. `outcomes` has events for 1,399 ticket IDs. Four of them (T-9990 to T-9993) are not in `tickets`. The extraction in the next lesson leaves them out.

Then Mei checks the ties. She uses `RANK()` and leaves out `outcome_id`.

> **Predict.** More rows, or the same? Then run the cell.

In [ ]:
sql("""
WITH ranked AS (
    SELECT
        ticket_id,
        RANK() OVER (
            PARTITION BY ticket_id
            ORDER BY recorded_at DESC
        ) AS rank_by_time
    FROM outcomes
)
SELECT
    COUNT(*) AS rows_with_rank_1,
    COUNT(DISTINCT ticket_id) AS tickets
FROM ranked
WHERE rank_by_time = 1;
""")

> **Check.** You should see **1432** rows for 1399 tickets. 33 tickets have more than one event at their latest time. Eight are like T-10136, reopened and resolved in the same second. The other 25 are retries: the same event logged twice, with a new `outcome_id`. `RANK()` shows every tie. To keep exactly one row, use `ROW_NUMBER()` and a column that breaks the tie.

## 4. Put each customer's tickets in order

Omar, the product manager, asks: "Do customers come back soon after a ticket?" Mei numbers each customer's tickets in time order, and computes the days since the previous ticket with `LAG`. On your computer, this is `queries/33-customer-history.sql`.

> **Predict.** What is `days_since_previous` in the first row? Then run the cell.

In [ ]:
sql("""
-- Each customer's tickets in time order, with the days since the previous ticket.
WITH unique_tickets AS (
    SELECT DISTINCT *
    FROM tickets
)
SELECT
    customer_id,
    ticket_id,
    created_at,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY created_at, ticket_id) AS nth_ticket,
    ROUND(
        julianday(created_at)
        - julianday(LAG(created_at) OVER (PARTITION BY customer_id ORDER BY created_at, ticket_id)),
        1
    ) AS days_since_previous
FROM unique_tickets
WHERE customer_id = 'C-0003'
ORDER BY customer_id, nth_ticket;
""")

> **Check.** You should see 5 rows for C-0003: T-10047, T-10391, T-10414, T-10598 and T-10877. `days_since_previous` is `NULL` in the first row, then 45.0, 2.2, 21.8 and 34.8. The first ticket has no previous ticket, so `LAG` is NULL, and the difference with NULL is NULL.

Check one difference by hand: from 10 January 12:00 to 24 February 13:05 is 45 days and about one hour, so 45.0.

> **Your turn: change one thing.** In the cell above, remove the line `WHERE customer_id = 'C-0003'`, and run it again. Now every customer has a partition, and `nth_ticket` starts at 1 for each one.

Two details in the query matter:

- **`ORDER BY created_at, ticket_id`** inside `OVER`. If one customer opens two tickets in the same second, `ticket_id` decides the order.
- **`WHERE` runs before the window.** A filter on `customer_id` is safe, because each customer is a whole partition. A filter on a date would remove earlier tickets, and then "the previous ticket" would be a different ticket.

## 5. Your turn: a customer with a repeated ticket

Customer C-0052 has a ticket that appears twice in the export. The query reads `tickets` directly. Run the next cell, and look at the second row: C-0052 seems to come back after 0 days, and to have 5 tickets.

> **Your turn.** Change the query so that each ticket counts once. Keep the four columns and the order. Then run the check cell. The check passes when the result has 4 rows in the right order.

In [ ]:
customer_c0052 = """
WITH unique_tickets AS (
    SELECT DISTINCT *
    FROM tickets
)
SELECT
    ticket_id,
    created_at,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY created_at, ticket_id) AS nth_ticket,
    ROUND(
        julianday(created_at)
        - julianday(LAG(created_at) OVER (PARTITION BY customer_id ORDER BY created_at, ticket_id)),
        1
    ) AS days_since_previous
FROM unique_tickets
WHERE customer_id = 'C-0052'
ORDER BY nth_ticket;
"""
sql(customer_c0052)

In [ ]:
check(customer_c0052, rows=4, columns=4, expected="4841061006c7d92a", ordered=True)

T-10609 was in the export twice. The copy became the customer's "2nd ticket", 0 days after the first. A feature such as "tickets in the last 90 days" would count it twice.

## 6. Failure case: tied events make resolved tickets look open

Tomás writes the latest-outcome query with `ORDER BY recorded_at DESC` only, with no `outcome_id`. The query returns one row per ticket, so the count check passes. Run his status summary.

In [ ]:
sql("""
WITH ranked AS (
    SELECT
        ticket_id,
        status,
        ROW_NUMBER() OVER (
            PARTITION BY ticket_id
            ORDER BY recorded_at DESC
        ) AS rn
    FROM outcomes
)
SELECT status, COUNT(*) AS tickets
FROM ranked
WHERE rn = 1
GROUP BY status
ORDER BY status;
""")

> **Check.** On the SQLite of this course, you see **9** tickets that end in `reopened`. Another SQLite version can show a different number, and that is the problem: for 8 tickets, `reopened` and `resolved` have the same `recorded_at`, so `ROW_NUMBER()` may number them in any order. The result is not reproducible.

Now run Mei's version, with `outcome_id DESC` to break the ties.

In [ ]:
sql("""
WITH ranked AS (
    SELECT
        ticket_id,
        status,
        ROW_NUMBER() OVER (
            PARTITION BY ticket_id
            ORDER BY recorded_at DESC, outcome_id DESC
        ) AS rn
    FROM outcomes
)
SELECT status, COUNT(*) AS tickets
FROM ranked
WHERE rn = 1
GROUP BY status
ORDER BY status;
""")

> **Check.** You should see `escalated` 3, `reopened` **1** and `resolved` 1395. Grace checked two of Tomás's "reopened" tickets in the help desk: they are resolved.

**Diagnose:** count the ties with `RANK()`, as in section 3: 1,432 rows with rank 1 for 1,399 tickets. **Fix:** add a column that makes the order unique and has a meaning: `outcome_id` follows the logging order, so the event logged last wins. Write this rule down, because Priya must know it.

> **Warning.** The latest status often becomes the target that a model learns to predict. Arbitrary ties put wrong targets on some rows, and a rerun of the same query can give a different training set.

## Check your understanding and recap

Answer the three **knowledge checks** at the end of the [lesson page](https://learning.nextia-ai.com/courses/sql/m03/time-and-ranking/). They count toward your certificate when you are signed in and enrolled.

In this lesson you learned that times in SQLite are text, that you check `MIN` and `MAX` before you trust a time column, and that half-open ranges (`>= start AND < end`) keep the last day. A window function adds a value to each row from its partition: `ROW_NUMBER()` numbers rows, `RANK()` shows ties, and `LAG()` reads the previous row. For "latest per ticket", order by time **and** by `outcome_id`, then check that rows = distinct tickets.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Reproducible extraction](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m03/time-and-ranking/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.